In [ ]:
import copy
import datetime as dt
from datetime import datetime
import importlib  # needed so that we can reload packages
import logging
import os
import pathlib
import sys
import time
import warnings
from typing import Union, Tuple
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.logger_utils import setup_clean_logger, mute_external_loggers

# SISEPUEDE imports
from sisepuede.manager.sisepuede_examples import SISEPUEDEExamples
from sisepuede.manager.sisepuede_file_structure import SISEPUEDEFileStructure
import sisepuede.core.support_classes as sc
import sisepuede.transformers as trf
import sisepuede.utilities._plotting as spu
import sisepuede.utilities._toolbox as sf
import sisepuede.core.attribute_table as att
import sisepuede.manager.sisepuede_examples as sxl
import sisepuede.manager.sisepuede_file_structure as sfs

# --- Runtime configuration ---
warnings.filterwarnings("ignore")

# Set up a clean logger for your notebook
logger = setup_clean_logger("notebook", logging.INFO)
logger.info("Notebook started successfully.")

# Mute logs from sisepuede to avoid duplication
mute_external_loggers(["sisepuede"])

In [ ]:
%load_ext autoreload
%autoreload 2

### Initial Set up

Make sure to edit the config yaml under ssp_modeling/config_files/config.yaml


In [ ]:
# Set up dir paths

CURR_DIR_PATH = pathlib.Path(os.getcwd())
SSP_MODELING_DIR_PATH = CURR_DIR_PATH.parent
PROJECT_DIR_PATH = SSP_MODELING_DIR_PATH.parent
DATA_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("input_data")
RUN_OUTPUT_DIR_PATH = SSP_MODELING_DIR_PATH.joinpath("ssp_run_output")
CONFIG_DIR_PATH = CURR_DIR_PATH.joinpath("config_files")

In [ ]:
from ssp_transformations_handler.GeneralUtils import GeneralUtils

# Initialize general utilities
g_utils = GeneralUtils()

In [ ]:
def get_file_structure(
    y0: int = 2015,
    y1: int = 2070,
) -> Tuple[sfs.SISEPUEDEFileStructure, att.AttributeTable]:
    """Get the SISEPUEDE File Structure and update the attribute table
        with new years.
    """
    file_struct = sfs.SISEPUEDEFileStructure(
        initialize_directories = False,
    )
 
    key_time_period = file_struct.model_attributes.dim_time_period
    key_year = file_struct.model_attributes.field_dim_year

    years = np.arange(y0, y1 + 1, ).astype(int)
    attribute_time_period = att.AttributeTable(
        pd.DataFrame(
            {
                key_time_period: range(len(years)),
                key_year: years,
            }
        ),
        key_time_period,
    )
 
    (
        file_struct
        .model_attributes
        .update_dimensional_attribute_table(
            attribute_time_period,
        )
    )
 
    return (file_struct, attribute_time_period, )


In [ ]:
# Set up model attributes (needed for subsector fields and color maps)
_FILE_STRUCTURE, _ATTRIBUTE_TABLE_TIME_PERIOD = get_file_structure(y1=2070)
matt = _FILE_STRUCTURE.model_attributes
regions = sc.Regions(matt, )

## Load specific run outputs

Reading directly from the run folder — no model execution needed.

In [ ]:
# ── Define run to analyze ─────────────────────────────────────────────────────
RUN_ID = "sisepuede_run_2026-03-10t13;27;53.264959/"
RUN_FOLDER_NAME = f"{RUN_ID}"
RUN_ID_OUTPUT_DIR_PATH = RUN_OUTPUT_DIR_PATH / RUN_FOLDER_NAME

print(f"Run folder: {RUN_ID_OUTPUT_DIR_PATH}")
print(f"Exists: {RUN_ID_OUTPUT_DIR_PATH.exists()}")

In [ ]:
# ── Read attribute tables ─────────────────────────────────────────────────────
att_primary  = pd.read_csv(RUN_ID_OUTPUT_DIR_PATH / "ATTRIBUTE_PRIMARY.csv")
att_strategy = pd.read_csv(RUN_ID_OUTPUT_DIR_PATH / "ATTRIBUTE_STRATEGY.csv")

print(f"att_primary  shape: {att_primary.shape}")
print(f"att_strategy shape: {att_strategy.shape}")
display(att_primary.head())
display(att_strategy)

In [ ]:
# ── Parse strategy metadata from strategy_code and strategy columns ──────────
#
# strategy_code patterns:
#   Singleton  : "TORNADO_BASE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ"             → sector=AGRC, transformation_code=DEC_CH4_RICE
#   WHIRLPOOL  : "WHIRLPOOL_PFLO_HBLE:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ" → sector=AGRC, transformation_code=DEC_CH4_RICE
#   Composite  : "AF:ALL"                        → sector=AF,   transformation_code=ALL
#   BASE       : "BASE"                          → NaN
#
# strategy text pattern (for singletons):
#   "Singleton - Default Value - AGRC: Improve rice management"

# sector + transformation_code from strategy_code (supports digits like DEC_CH4_RICE)
att_strategy['sector'] = (
    att_strategy['strategy_code']
    .str.extract(r'(?:[A-Z0-9_]+:TX:)?([A-Z]+):[A-Z0-9_]+$', expand=False)
)

att_strategy['transformation_code'] = (
    att_strategy['strategy_code']
    .str.extract(r':([A-Z0-9_]+)$', expand=False)
    .str.replace(r'_STRATEGY_NZ$', '', regex=True)
)

# transformation_name from the strategy description text
att_strategy['transformation_name'] = (
    att_strategy['strategy']
    .str.extract(r'-\s*[A-Z]+:\s*(.+)$', expand=False)
)

# combined field: "AGRC: Improve rice management"
att_strategy['transformation_name_sector'] = (
    att_strategy['sector'].fillna('')
    + ': '
    + att_strategy['transformation_name'].fillna('')
).where(att_strategy['transformation_name'].notna())

display(att_strategy[['strategy_id','strategy_code','sector','transformation_code',
                        'transformation_name','transformation_name_sector']])

In [ ]:
# ── Read wide inputs/outputs CSV ──────────────────────────────────────────────
_wide_csv = list(RUN_ID_OUTPUT_DIR_PATH.glob("*c585e7e9-e32f-4131-999b-ee7fc5ec014e.csv"))
assert _wide_csv, f"No se encontró archivo WIDE en {RUN_ID_OUTPUT_DIR_PATH}"

df_export = pd.read_csv(_wide_csv[0])
KEY_PRIMARY = "primary_id"

# Filter to keep only primary_id 0 (baseline) and 2002 (strategy 2)
df_export = df_export[df_export[KEY_PRIMARY].isin([
    0, 760076, 770077, 780078, 790079, 800080, 810081, 820082, 830083,
    840084, 850085, 860086, 870087, 880088, 890089, 900090, 910091,
    920092, 930093, 940094, 950095, 960096, 970097, 980098, 990099,
    1000100, 1010101, 1020102, 1030103, 1040104, 1050105, 1060106,
    1070107, 1080108, 1090109, 1100110, 1110111, 1120112, 1130113,
    1140114, 1150115, 1160116, 1170117, 1180118, 1190119, 1200120,
    1210121, 1220122, 1230123, 1240124, 1250125, 1260126, 1270127,
    1280128, 1290129, 1300130, 1310131, 1320132, 1330133, 1340134
])]

print(f"df_export shape: {df_export.shape}")
df_export[[KEY_PRIMARY, 'region', 'time_period']].drop_duplicates().head(10)

## Emissions Stack Plots

In [ ]:
def plot_field_stack(
    df,
    fields,
    dict_format,
    time_col="time_period",
    primary_id=0,
    figsize=(18, 8),
    legend_loc='upper right',
    legend_bbox=(1.1, 1),
    ylabel="MT Emissions CO2e",
    xlabel="Time Period",
    title=None,
):
    """
    Plots a stack plot of the selected fields for a given primary_id.

    Args:
        df (pd.DataFrame): DataFrame containing output data.
        fields (list): List of column names to plot.
        dict_format (dict): Formatting dictionary for colors.
        time_col (str): Name of the time column.
        primary_id (int): Value of primary_id to filter.
        figsize (tuple): Figure size.
        legend_loc (str): Legend location.
        legend_bbox (tuple): Legend bbox_to_anchor.
        ylabel (str): Y-axis label.
        xlabel (str): X-axis label.
        title (str): Plot title.
    """
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)

    df_plot = df[df[KEY_PRIMARY].isin([primary_id])]

    fig, ax = spu.plot_stack(
        df_plot,
        fields,
        dict_formatting=dict_format,
        field_x=time_col,
        figtuple=(fig, ax),
    )

    ax.legend(loc=legend_loc, bbox_to_anchor=legend_bbox, title="Fields")
    plt.show()

In [ ]:
# Define the fields to plot and the formatting dictionary
subsector_emission_fields = matt.get_all_subsector_emission_total_fields()

dict_format = dict(
    (k, {"color": v}) for (k, v) in
    matt.get_subsector_color_map().items()
)

In [ ]:
primary_ids_to_plot = df_export[KEY_PRIMARY].unique()

In [ ]:
# Plot the emissions stack for each primary_id
for primary_id in primary_ids_to_plot:

    plot_field_stack(
        df_export,
        subsector_emission_fields,
        dict_format,
        primary_id=primary_id,
        title=f"Emissions Stack Plot for Primary ID {primary_id}"
    )

## Post-procesamiento: Intertemporal Decomposition (rescaling)


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR_PATH))

from ssp_modeling.output_postprocessing.intertemporal_decomposition import (
    run_postprocessing,
    prepare_targets,
    preprocess_ssp_output,
    rescale,
)
print("Módulo de post-procesamiento cargado correctamente.")

In [ ]:
# ── Parámetros ────────────────────────────────────────────────────────────────
TARGETS_PATH = PROJECT_DIR_PATH / "ssp_modeling/output_postprocessing/data/LULUCF/emission_targets_uganda_2019_LULUCF.csv"
ISO_CODE3    = "UGA"
YEAR_REF     = 2019
REGION       = 'uganda'          # "libya"  ← viene del config.yaml

# Ruta donde guardar el CSV resultante (None = no escribir)
OUTPUT_PATH  = Path(RUN_ID_OUTPUT_DIR_PATH) / "decomposed_ssp_output_tornado.csv"

print(f"Targets  : {TARGETS_PATH}")
print(f"ISO code : {ISO_CODE3}")
print(f"Year ref : {YEAR_REF}")
print(f"Region   : {REGION}")
print(f"Output   : {OUTPUT_PATH}")

In [ ]:
# ── Ejecutar rescaling ────────────────────────────────────────────────────────
df_decomposed = run_postprocessing(
    df_ssp_output         = df_export,
    targets_path          = TARGETS_PATH,
    iso_code3             = ISO_CODE3,
    year_ref              = YEAR_REF,
    region                = REGION,
    initial_conditions_id = "_0",
    output_path           = OUTPUT_PATH,
)

print(f"\nResultado: {df_decomposed.shape[0]} filas × {df_decomposed.shape[1]} columnas")
df_decomposed.head(3)

### Tests de validación

Compara los resultados Python con el CSV que generó el script R.
Si el CSV de R no existe, se corren validaciones internas de consistencia.

## Costos y Beneficios

Corre el pipeline de costos y beneficios sobre `df_decomposed` (ya en memoria).
Equivalente a `ssp_modeling/cost-benefits/cb.ipynb` pero sin escribir CSVs intermedios.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR_PATH / 'ssp_modeling' / 'cost-benefits'))
from costs_benefits_ssp.cb_calculate import CostBenefits

CB_CONFIG_PATH = PROJECT_DIR_PATH / 'ssp_modeling' / 'cost-benefits' / 'cb_cost_factors' / 'cb_config_params.xlsx'
CB_OUTPUT_PATH = PROJECT_DIR_PATH / 'ssp_modeling' / 'cost-benefits' / 'out'
CB_OUTPUT_PATH.mkdir(exist_ok=True)

print(f'att_primary : {att_primary.shape}')
print(f'att_strategy: {att_strategy.shape}')

In [ ]:
CB_CONFIG_PATH

In [ ]:
#Strategy baaleline to CBA
strategy_code_base = 'BASE'
if strategy_code_base not in att_strategy['strategy_code'].values:
    raise ValueError(f"Base strategy '{strategy_code_base}' no encontrada en att_strategy")

# Usa df_decomposed que ya está en memoria (resultado del rescaling)
cb = CostBenefits(df_decomposed, att_primary, att_strategy, strategy_code_base)



In [ ]:
cb.load_cb_parameters(str(CB_CONFIG_PATH))

In [ ]:
results_system = cb.compute_system_cost_for_all_strategies(verbose=False)

In [ ]:
results_tx = cb.compute_technical_cost_for_all_strategies(verbose=False)

In [ ]:
results_all    = pd.concat([results_system, results_tx], ignore_index=True)

results_all_pp         = cb.cb_process_interactions(results_all)
results_all_pp_shifted = cb.cb_shift_costs(results_all_pp)

print(f'results_all_pp_shifted: {results_all_pp_shifted.shape}')
results_all_pp_shifted.head(3)

In [ ]:
# ── Reshape para análisis / Tableau ──────────────────────────────────
cb_data = results_all_pp_shifted.copy()

# Split variable en partes (name:sector:cb_type:item_1:item_2)
cb_chars = cb_data['variable'].astype(str).str.split(':', n=4, expand=True)
cb_chars.columns = ['name', 'sector', 'cb_type', 'item_1', 'item_2']
cb_data = pd.concat([cb_data, cb_chars], axis=1)

# Escalar de USD a billones (B USD)
cb_data['value'] = cb_data['value'] / 1e9
cb_data['variable_value_baseline'] = cb_data['variable_value_baseline'] / 1e9
cb_data['variable_value_pathway'] = cb_data['variable_value_pathway'] / 1e9

# Eliminar filas 'shifted'
cb_data = cb_data[~cb_data['item_2'].astype(str).str.contains('shifted', na=False)]
cb_data = cb_data[~cb_data['variable'].astype(str).str.contains('shifted2', na=False)]

# Año calendario
cb_data['Year'] = cb_data['time_period'] + 2015

# ── Mapeos dinámicos desde att_strategy y att_primary ────────────────
strategy_id_map   = att_strategy.set_index('strategy_code')['strategy_id'].to_dict()
strategy_name_map = att_strategy.set_index('strategy_code')['strategy'].to_dict()

# primary_id por estrategia: une att_primary con att_strategy en strategy_id
primary_id_map = (
    att_strategy[['strategy_code', 'strategy_id']]
    .merge(att_primary[['primary_id', 'strategy_id']], on='strategy_id', how='left')
    .drop_duplicates('strategy_code')
    .set_index('strategy_code')['primary_id']
    .to_dict()
)

cb_data['strategy']    = cb_data['strategy_code'].astype(str).map(strategy_name_map).fillna(cb_data['strategy_code'])
cb_data['strategy_id'] = cb_data['strategy_code'].astype(str).map(strategy_id_map)
cb_data['primary_id']  = cb_data['strategy_code'].astype(str).map(primary_id_map)

# IDs compuestos
cb_data['ids'] = cb_data['variable'].astype(str) + ':' + cb_data['strategy_id'].astype(str)

# Corregir signo capex electricidad (debe ser negativo = inversión)
mask_entc = cb_data['variable'].astype(str).str.contains('cb:entc:technical_cost:electricity:capex', na=False)
cb_data.loc[mask_entc, 'value'] = -cb_data.loc[mask_entc, 'value'].abs()

# Merge GDP
gdp = df_decomposed[['primary_id', 'time_period', 'gdp_mmm_usd']].copy()
cb_data = cb_data.merge(gdp, on=['primary_id', 'time_period'], how='left')

print(f'cb_data: {cb_data.shape}   NaNs en value: {cb_data["value"].isna().sum()}')
cb_data.head(3)

In [ ]:
cb_data.to_csv(Path(RUN_ID_OUTPUT_DIR_PATH) / "cost_benefits_data_tornado.csv", index=False, encoding="UTF-8")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# ── 1. Filtrar: solo filas con al menos un valor no-cero (baseline o pathway) ──
df_viz = cb_data[
    (cb_data['variable_value_baseline'] != 0) | (cb_data['variable_value_pathway'] != 0)
].copy()

# ── 2. Agregar por (cb_type, difference_variable, strategy_id, Year) ───────────
agg = (
    df_viz
    .groupby(['cb_type', 'difference_variable', 'strategy_id', 'Year'], dropna=False)[
        ['variable_value_baseline', 'variable_value_pathway']
    ]
    .sum()
    .reset_index()
)

# Mantener solo combos (cb_type, difference_variable) que sean != 0
# en TODOS los periodos (usamos la lógica del análisis anterior)
all_years = set(agg['Year'].unique())
nonzero_check = (
    agg.groupby(['cb_type', 'difference_variable', 'strategy_id'])
    .apply(lambda g: (g['variable_value_baseline'] != 0).all() or (g['variable_value_pathway'] != 0).all())
    .reset_index(name='always_nonzero')
)
valid_combos = nonzero_check[nonzero_check['always_nonzero']][['cb_type', 'difference_variable', 'strategy_id']]
agg = agg.merge(valid_combos, on=['cb_type', 'difference_variable', 'strategy_id'], how='inner')

cb_types = sorted(agg['cb_type'].dropna().unique())
strategy_ids = sorted(agg['strategy_id'].dropna().unique())

print(f"cb_types activos   : {cb_types}")
print(f"strategy_ids únicos: {strategy_ids}")
print(f"Filas en agg       : {len(agg)}")

In [ ]:
# ── 3. Diferencia (value) por difference_variable en el tiempo ────────────────
CB_TYPE_FILTER = ['technical_cost']   # ← cambia aquí para incluir otros cb_types

PALETTE = [
    '#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
    '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
]

# Agregar value por (strategy_id, difference_variable, Year)
agg_val = (
    cb_data[cb_data['cb_type'].isin(CB_TYPE_FILTER) & (cb_data['value'] != 0)]
    .groupby(['strategy_id', 'difference_variable', 'Year'], dropna=False)['value']
    .sum()
    .reset_index()
)

print(f'Filas en agg_val: {len(agg_val)}')
print(f'strategy_ids: {sorted(agg_val["strategy_id"].unique())}')
print(f'difference_variables: {sorted(agg_val["difference_variable"].dropna().unique())}')
display(agg_val.head(10))

for strat_id in sorted(agg_val['strategy_id'].unique()):
    sub = agg_val[agg_val['strategy_id'] == strat_id].copy()
    if sub.empty:
        continue

    strategy_label = cb_data.loc[cb_data['strategy_id'] == strat_id, 'strategy'].iloc[0]
    diff_vars = sorted(sub['difference_variable'].dropna().unique())

    fig = go.Figure()
    for i_dv, dv in enumerate(diff_vars):
        df_dv = sub[sub['difference_variable'] == dv].sort_values('Year')
        color = PALETTE[i_dv % len(PALETTE)]
        short_dv = dv if len(dv) <= 45 else dv[:42] + '...'
        fig.add_trace(go.Scatter(
            x=df_dv['Year'], y=df_dv['value'],
            mode='lines+markers',
            line=dict(color=color, width=2),
            marker=dict(size=5),
            name=short_dv,
            hovertemplate=f'<b>{dv}</b><br>value: %{{y:,.2f}}<br>Year: %{{x}}<extra></extra>',
        ))

    fig.add_hline(y=0, line=dict(color='black', dash='dash', width=1))
    fig.update_layout(
        title=dict(
            text=(
                f'<b>Strategy {int(strat_id)}: {strategy_label}</b><br>'
                f'<sup>cb_type: {", ".join(CB_TYPE_FILTER)} · value = pathway − baseline</sup>'
            ),
            x=0.5,
        ),
        xaxis_title='Year', xaxis=dict(tickformat='d'),
        yaxis_title='value',
        height=500, width=1100,
        legend=dict(orientation='v', x=1.01, y=1, font=dict(size=9)),
        hovermode='x unified',
    )
    fig.show()

In [ ]:
# ── Costos técnicos por año y por estrategia ──────────────────────────────────
import plotly.graph_objects as go

tc_annual = (
    cb_data[cb_data['cb_type'] == 'technical_cost']
    .groupby(['strategy_id','strategy','Year'], dropna=False)['value']
    .sum()
    .reset_index()
    .sort_values(['strategy_id','Year'])
)

PALETTE = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
           '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']

fig_tc = go.Figure()
for i, (sid, grp) in enumerate(tc_annual.groupby('strategy_id', sort=True)):
    label = grp['strategy'].iloc[0]
    color = PALETTE[i % len(PALETTE)]
    dash  = 'dash' if sid == 0 else 'solid'
    fig_tc.add_trace(go.Scatter(
        x=grp['Year'], y=grp['value'],
        mode='lines+markers',
        line=dict(color=color, dash=dash, width=2),
        marker=dict(size=5),
        name=label,
        hovertemplate=f'<b>{label}</b><br>%{{y:,.3f}} B USD<br>Year: %{{x}}<extra></extra>',
    ))

fig_tc.add_hline(y=0, line=dict(color='black', dash='dot', width=1))
fig_tc.update_layout(
    title=dict(text='<b>Technical cost por estrategia</b><br>'
                    '<sup>B USD · línea punteada = baseline</sup>', x=0.5),
    xaxis=dict(title='Year', tickformat='d'),
    yaxis_title='B USD',
    height=500, width=1000,
    legend=dict(orientation='v', x=1.01, y=1, font=dict(size=9)),
    hovermode='x unified',
)
fig_tc.show()

## MAC – Paso 1: Mapeo de emisiones reescaladas a categorías de inventario

Equivalente a `data_prep_new_mapping.r` aplicado sobre `df_decomposed`.

In [ ]:
# ── 0. Dependencias ──────────────────────────────────────────────────────────
try:
    from statsmodels.tsa.filters.hp_filter import hpfilter as sm_hpfilter
    HAS_SM = True
except ImportError:
    HAS_SM = False
    print('AVISO: statsmodels no disponible. Instala con: pip install statsmodels')

INVENT_DIR = PROJECT_DIR_PATH / 'ssp_modeling/output_postprocessing/data/LULUCF'

# ── 1. Mapping de inventario ─────────────────────────────────────────────────
mapping = pd.read_csv(INVENT_DIR / 'emission_targets_uganda_2019_LULUCF.csv')
# Normalizar nombres de columnas (nuevo formato LULUCF → nombres esperados)
mapping = mapping.rename(columns={
    'ssp_subsector':     'subsector_ssp',
    'Gas':               'gas',
    'Vars':              'vars',
    'Subsector_Category':'ID',
    'Subsector':         'subsector',
})
# El nuevo CSV no tiene columna 'sector'; usar el código SSP como sector
if 'sector' not in mapping.columns:
    mapping['sector'] = mapping['subsector_ssp']
mapping = mapping.drop(columns=[ISO_CODE3, 'est_from_sisepuede'], errors='ignore').reset_index(drop=False).rename(columns={'index':'row_idx'})
mapping['ids'] = mapping['row_idx'].astype(str) + ':' + mapping['subsector_ssp'].astype(str) + ':' + mapping['gas'].astype(str)

# ── 2. EDGAR histórico ───────────────────────────────────────────────────────
edgar = pd.read_csv(INVENT_DIR / 'inventory_trajectories.csv')
edgar = edgar.rename(columns={'CSC.Sector': 'sector', 'CSC.Subsector': 'subsector'})
edgar = edgar[edgar['Code'] == ISO_CODE3].copy()
edgar['ID'] = edgar['Subsector_Category']  # ya tiene formato 'subsector:gas'
year_cols = [c for c in edgar.columns if str(c).isdigit()]
edgar_long = edgar.melt(id_vars=['Code','sector','subsector','Gas','ID'],
                         value_vars=year_cols, var_name='year_str', value_name='value')
edgar_long['Year'] = edgar_long['year_str'].astype(int)
edgar_long = edgar_long.drop(columns=['year_str'])
for col in ['strategy_id','primary_id','design_id','future_id']:
    edgar_long[col] = float('nan')
edgar_long['strategy'] = 'Historical'
edgar_long['source']   = 'EDGAR'
edgar_long['Contry']   = REGION
edgar_max_year = int(edgar_long['Year'].max())

# ── 3. Mapear vars SSP a categorías de inventario ────────────────────────────
id_vars = ['region','time_period','primary_id']
data = df_decomposed[df_decomposed['region'] == REGION].copy()

rows_agg = []
for _, row in mapping.iterrows():
    tvars = [v.strip() for v in str(row['vars']).split(':') if v.strip() in data.columns]
    if len(tvars) > 1:
        agg_col = data[tvars].sum(axis=1)
    elif len(tvars) == 1:
        agg_col = data[tvars[0]]
    else:
        agg_col = pd.Series(0.0, index=data.index)
    tmp = data[id_vars].copy()
    tmp['ids']   = row['ids']
    tmp['value'] = agg_col.values
    rows_agg.append(tmp)

data_long = pd.concat(rows_agg, ignore_index=True)

# ── 4. Merge metadatos + agregación a nivel inventario ───────────────────────
meta = mapping[['ids','sector','subsector','gas','ID']].copy()
data_long = data_long.merge(meta, on='ids', how='left')

data_inv = (
    data_long
    .groupby(['primary_id','time_period','ID','sector','subsector'], dropna=False)['value']
    .sum()
    .reset_index()
)
data_inv['Year']   = data_inv['time_period'] + 2015
data_inv['Gas']    = data_inv['ID'].str.split(':').str[-1]
data_inv['Code']   = ISO_CODE3
data_inv['Contry'] = REGION
data_inv['source'] = 'SISEPUEDE'

# ── 5. Merge estrategias + filtrar años >= edgar_max_year ────────────────────
data_inv = data_inv.merge(att_primary[['primary_id','strategy_id','design_id','future_id']],
                           on='primary_id', how='left')
data_inv = data_inv.merge(att_strategy[['strategy_id','strategy']], on='strategy_id', how='left')
data_inv = data_inv[data_inv['Year'] >= edgar_max_year].copy()

# ── 6. Combinar SSP + EDGAR ──────────────────────────────────────────────────
shared = ['primary_id','strategy_id','design_id','future_id',
          'sector','subsector','Gas','ID','Year','value','Code','Contry','strategy','source']
emissions = pd.concat(
    [data_inv[[c for c in shared if c in data_inv.columns]],
     edgar_long[[c for c in shared if c in edgar_long.columns]]],
    ignore_index=True
)
emissions = emissions.sort_values(['strategy_id','sector','subsector','Gas','Year']).reset_index(drop=True)

print(f'emissions shape: {emissions.shape}')
print(f'Años SSP  : {sorted(data_inv["Year"].unique())}')
print(f'Estrategias: {emissions["strategy"].dropna().unique()}')
display(emissions.head(8))

In [ ]:
# ── Emisiones acumuladas por estrategia (2023 – sim_end_year) ────────────────
em_ssp = emissions[emissions['source'] == 'SISEPUEDE'].copy()

cumul = (
    em_ssp
    .groupby(['strategy_id','primary_id'], dropna=False)['value']
    .sum()
    .reset_index()
    .rename(columns={'value': 'emission_total'})
    .sort_values('strategy_id')
)

cumul['emission_total'] = cumul['emission_total'].round(4)

cumul.head()

In [ ]:
# -- Baseline strategy ID --
base_sid = att_strategy.loc[att_strategy['strategy_code'] == 'BASE', 'strategy_id'].iloc[0]
base_val = cumul.loc[cumul['strategy_id'] == base_sid, 'emission_total'].values[0]

cumul['base_emission_total'] = base_val
cumul['emission_diff'] =  cumul['emission_total'] - base_val 

print('Emisiones acumuladas totales (MtCO2e) por estrategia:')
cumul.head()

In [ ]:
# ── Emisiones totales apiladas por subsector · una figura por estrategia ─────
em_stack = (
    em_ssp
    .groupby(['strategy_id','strategy','subsector','Year'], dropna=False)['value']
    .sum()
    .reset_index()
    .sort_values(['strategy_id','subsector','Year'])
)

subsectors_all = sorted(em_stack['subsector'].dropna().unique())  # A→Z; CCSQ al final

PALETTE = [
    '#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
    '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
    '#aec7e8','#ffbb78','#98df8a','#ff9896','#c5b0d5',
]
color_map = {sub: PALETTE[i % len(PALETTE)] for i, sub in enumerate(subsectors_all)}

for sid, strat_grp in em_stack.groupby('strategy_id', sort=True):
    strategy_label = strat_grp['strategy'].iloc[0]
    fig = go.Figure()
    seen = set()
    for sub in reversed(subsectors_all):  # reversed → legend A→Z, CCSQ al final
        df_sub = strat_grp[strat_grp['subsector'] == sub].sort_values('Year')
        if df_sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=df_sub['Year'], y=df_sub['value'],
            mode='lines',
            stackgroup='one',
            name=sub,
            line=dict(color=color_map[sub], width=0.5),
            fillcolor=color_map[sub],
            hovertemplate=f'<b>{sub}</b><br>%{{y:,.2f}} MtCO2e<br>Year: %{{x}}<extra></extra>',
        ))
    fig.update_layout(
        title=dict(
            text=f'<b>{strategy_label}</b><br><sup>Emisiones apiladas por subsector (MtCO2e)</sup>',
            x=0.5,
        ),
        xaxis=dict(title='Year', tickformat='d'),
        yaxis_title='MtCO2e',
        height=500, width=1000,
        legend=dict(orientation='v', x=1.01, y=1, font=dict(size=9)),
        hovermode='x unified',
    )
    fig.show()
# ── Total de emisiones por estrategia (líneas) ───────────────────────────────
em_total = (
    em_ssp
    .groupby(['strategy_id','strategy','Year'], dropna=False)['value']
    .sum()
    .reset_index()
    .sort_values(['strategy_id','Year'])
)

fig_total = go.Figure()
for i, (sid, grp) in enumerate(em_total.groupby('strategy_id', sort=True)):
    label = grp['strategy'].iloc[0]
    color = PALETTE[i % len(PALETTE)]
    dash  = 'dash' if sid == base_sid else 'solid'
    fig_total.add_trace(go.Scatter(
        x=grp['Year'], y=grp['value'],
        mode='lines+markers',
        line=dict(color=color, dash=dash, width=2),
        marker=dict(size=5),
        name=label,
        hovertemplate=f'<b>{label}</b><br>%{{y:,.2f}} MtCO2e<br>Year: %{{x}}<extra></extra>',
    ))

fig_total.add_hline(y=0, line=dict(color='black', dash='dot', width=1))
fig_total.update_layout(
    title=dict(text='<b>Emisiones totales por estrategia</b><br><sup>MtCO2e · Linea punteada = baseline</sup>', x=0.5),
    xaxis=dict(title='Year', tickformat='d'),
    yaxis_title='MtCO2e',
    height=500, width=1000,
    legend=dict(orientation='v', x=1.01, y=1, font=dict(size=10)),
    hovermode='x unified',
)
fig_total.show()

## MAC – Paso 2: Costos técnicos acumulados por estrategia

In [ ]:
# ── Technical costs acumulados por estrategia (desde cb_data) ────────────────
# cb_data['value'] ya está en B USD (se escaló /1e9 en la celda de cb_data)

tc = (
    cb_data[cb_data['cb_type'] == 'technical_cost']
    .groupby(['strategy_id','primary_id'], dropna=False)['value']
    .sum()
    .reset_index()
    .rename(columns={'value': 'technical_cost'})
    .sort_values('strategy_id')
)

tc['technical_cost'] = tc['technical_cost']* -1

tc.head()

In [ ]:
# ── Unir con tabla de emisiones acumuladas ────────────────────────────────────
mac_df = cumul.merge(tc, on=['strategy_id','primary_id'], how='left')
mac_df.head()

In [ ]:
# Add parsed strategy metadata
mac_df = mac_df.merge(
    att_strategy[['strategy_id','sector','transformation_code']],
    on='strategy_id',
    how='left'
)

# Exclude baseline (strategy_id == 0)
mac_df = mac_df[mac_df['strategy_id'] != 0]

mac_df['emission_diff']  = round(mac_df['emission_diff'],  0)
mac_df['technical_cost'] = round(mac_df['technical_cost'], 4)

mac_df.head()

In [ ]:
# MAC = Marginal Abatement Cost
# Unidades: B USD / MtCO2e → USD / tCO2e

mac_df['marginal_abatement_cost'] = (
    (mac_df['technical_cost'] * 1e9)   # B USD → USD
    / (mac_df['emission_diff'] * 1e6)   # MtCO2e → tCO2e
)

# Preserve the sign of technical_cost
mac_df['marginal_abatement_cost'] = (
    mac_df['marginal_abatement_cost'].abs()
    * np.sign(mac_df['technical_cost'])
)

# Reorder columns
first_cols = ['strategy_id', 'primary_id', 'sector', 'transformation_code']
rest_cols  = [c for c in mac_df.columns if c not in first_cols]
mac_df = mac_df[first_cols + rest_cols]

mac_df.head()

In [ ]:
mac_df.to_csv(Path(RUN_ID_OUTPUT_DIR_PATH) / "marginal_abatement_costs.csv", index=False, encoding="UTF-8")